[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/quizzes/quiz_06_pytorch.ipynb)

# 🧪 Module 6 Quiz — Deep Learning with PyTorch

Five recall questions to check what stuck after Module 6 (the three PyTorch notebooks — fundamentals, training craft, and embeddings/serving).

- **Format:** multiple choice; each question has one correct answer.
- **Time:** ~10 minutes.
- **How to use:** read the question, decide which answer you think is correct, *then* click the **Answer + reasoning** block. Don't peek.

---

### Q1 — The loss that lurches

A colleague's training loop runs `loss.backward()` and `optimizer.step()` every epoch — nothing else — and the loss falls for a while, then jumps back up, and never settles. What's wrong?

- **A.** The learning rate is too high — halve it until the curve smooths out.
- **B.** `optimizer.zero_grad()` is missing: gradients **accumulate** across `.backward()` calls, so each step follows the *sum* of every gradient so far instead of the current one.
- **C.** The model needs `model.train()` before the loop; without it, gradients aren't computed.
- **D.** `loss.backward()` must be called *after* `optimizer.step()`, not before.

<details>
<summary>💡 <b>Answer + reasoning</b></summary>

**Correct answer: B.**

PyTorch gradients **add into `.grad`** on every `.backward()` — by design (it makes gradient accumulation across sub-batches trivial) — so *you* must wipe them each step with `optimizer.zero_grad()`. Forget it and by epoch 20 each "gradient" is really twenty stale gradients summed: the steps chase an outdated direction and the loss lurches instead of settling. The five-step loop is forward → loss → **backward → step → zero**; the learning rate (A) was innocent, `model.train()` (C) toggles dropout/batch-norm modes rather than gradient computation, and the backward-then-step order (D) is already correct.
</details>

### Q2 — Two curves, one decision

You log train loss and validation loss per epoch. Both fall together for 30 epochs; then train loss keeps falling toward zero while validation loss bottoms out and climbs. What is happening, and what's the right response?

- **A.** The model is underfitting — add layers and train longer until the train loss reaches zero.
- **B.** The validation set must be broken — data this consistent can't get worse with more training.
- **C.** Classic overfitting: past the valley the model is memorising training rows. Stop around the validation minimum (early stopping keeps a snapshot of the best weights) and/or add regularisation like dropout or weight decay.
- **D.** Normal behaviour — only the training loss matters, since that's what the optimizer minimises.

<details>
<summary>💡 <b>Answer + reasoning</b></summary>

**Correct answer: C.**

The parting of the curves is *the* diagnostic picture of deep learning: once validation loss turns up, every further epoch makes the model better at reciting the training set and worse at its actual job. The gap between the curves is the size of the lie the train loss is telling. Treatments: **early stopping** (track the best validation loss, snapshot the weights, restore them at the end), **dropout**, **weight decay** — each a dial you set against *validation* data, never the test set, which stays in the vault until one final look.
</details>

### Q3 — The coin-flipping model

A dropout model gives a *different* churn probability every time you score the **same** customer, and validation loss looks worse than training loss ever was. What's the fix?

- **A.** Call `model.eval()` (and wrap scoring in `torch.no_grad()`): the model was left in training mode, so dropout kept randomly zeroing hidden units at inference.
- **B.** Set a random seed before every prediction so dropout picks the same units each time.
- **C.** Lower the learning rate — noisy predictions mean the optimizer overshot.
- **D.** Remove the sigmoid; probabilities should be read straight from the logits.

<details>
<summary>💡 <b>Answer + reasoning</b></summary>

**Correct answer: A.**

Dropout is a **training-time** regulariser: in `train()` mode it randomly zeroes units on every forward pass — which is exactly what you want while learning and exactly what you don't while predicting. `model.eval()` switches it off (and fixes batch-norm statistics too), making predictions deterministic; `torch.no_grad()` additionally stops autograd from recording a graph nobody will backpropagate through. Seeding every prediction (B) would hide the bug, not fix it — you'd get *reproducibly wrong* outputs from a randomly thinned network.
</details>

### Q4 — One-hot vs. `nn.Embedding`

Your churn table gains a `customer_city` column with 5,000 distinct values, and next quarter new cities will appear that training never saw. What's the right treatment in a neural model?

- **A.** One-hot encode it — 5,000 extra columns is fine, and unseen cities will just be all-zeros rows.
- **B.** Drop the column; high-cardinality categoricals can't be used in neural networks.
- **C.** Ordinal-encode it (city → 0..4999) and feed it in as one scaled numeric feature.
- **D.** An `nn.Embedding(5001, 50)`: each city becomes a 50-dim *learned* vector (similar cities end up near each other), with **index 0 reserved for unknown** so unseen cities map to a harmless row instead of crashing.

<details>
<summary>💡 <b>Answer + reasoning</b></summary>

**Correct answer: D.**

An embedding is a trainable lookup table — `weight[idx]`, nothing more — and it's built for exactly this regime: 5,000 one-hot columns in every row (A) explode the data and learn no similarity structure, while an ordinal code (C) invents a fake numeric order (city 4999 isn't "more" than city 12). The two conventions that make embeddings production-safe: size the table **cardinality + 1** and reserve **index 0 for UNK** at training time, so a brand-new city looks up a real, neutral row — versus a `KeyError` (vocab miss) or `IndexError` (out-of-range index) when Sales launches in a new market. Rule of thumb for the width: `min(50, (cardinality + 1) // 2)`.
</details>

### Q5 — Should this be deep learning?

A stakeholder asks you to model churn on **8,000 rows** of clean tabular data (a handful of numeric features, two low-cardinality categoricals). What's the professional first move — and when *would* a neural network be the right call?

- **A.** Start with an LLM — it's the most advanced model, so it will beat the classical ones.
- **B.** Fit a logistic-regression baseline and gradient boosting first; reach for an embedding network only with a reason — high-cardinality categoricals, extra modalities (text/images), a need for reusable embeddings or transfer learning, or much more data.
- **C.** Always the neural network: with enough epochs it can approximate any function, so it can't lose.
- **D.** Skip modelling and use the rule "many support tickets = churn" — models are overkill at this size.

<details>
<summary>💡 <b>Answer + reasoning</b></summary>

**Correct answer: B.**

Module 6's bake-offs made the point empirically: on small, well-behaved tables the thirty-second baseline and gradient boosting tie or beat an MLP — the universal-approximation argument (C) says a network *can* fit anything, not that it will *generalise* better from 8,000 rows. Deep learning is a **capability, not an obligation**: it earns its keep through embeddings for high-cardinality categories, mixed modalities, transfer learning, representation reuse, and scale. The professional posture is a scoreboard, not a vibe — same split, same metric, simplest model wins ties.
</details>

---

## ✅ Done

Score yourself out of 5 (see the [rubric](README.md#scoring-rubric)).

**Review:** the three notebooks in [`../06_pytorch/`](../06_pytorch/) — [`20_pytorch_fundamentals.ipynb`](../06_pytorch/20_pytorch_fundamentals.ipynb), [`21_training_neural_networks.ipynb`](../06_pytorch/21_training_neural_networks.ipynb), and [`22_pytorch_in_practice.ipynb`](../06_pytorch/22_pytorch_in_practice.ipynb).

**Next:** [Module 12 — DeepTab](../12_deeptab/) industrialises NB 22's architecture; the [Module 5 appendices A2–A3](../05_machine_learning/) take PyTorch to images, sequences and fine-tuned transformers.